In [4]:
# Extract_ABMC_Graves_FAG.py
# 6/5/2025

# fileIn="FindAgrave_ABMC_Cemeteries"
# fileOutput="FindAgrave_ABMC_Graves"
# fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
# fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time_pause,sleep_range
    if call_count%100==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    try:
        collection_df['links']=collection_df['links'].str[0].str[2]
    except:
        collection_df['links']=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

blank_line='                                                                                                                                                                                           '


#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract_ABMC_Graves_FAG'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileIn="FindAgrave_ABMC_Cemeteries"
fileOutput="FindAgrave_ABMC_Graves"
if append==1:fileInputEvents='FindAgrave_ABMC_Cemeteries_X'

fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"OutputData\\Graves\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Graves\\"+fileOutput+"_Interim.csv"

cemetery_frame_df_header=['cemetery_id','memorial_count','cemetery','city','county','state','state_id','county_id']
cemetery_info_df=pd.read_csv(fileCemeteriesIn)


field_list=['lastName','firstName','middleName','fullName','nickName','birthDate','birthMonth','birthYear','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links', 'intermentHasPhoto','Parents','Spouses','Siblings','Children','hasPlot','isMemorial','plot']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=2*60

url_cemetery_id = "https://www.findagrave.com/cemetery/"
url_skip ="/memorial-search?ajax=true&skip="
url_birth_year ="&limit=100&firstname=&middlename=&lastname=&birthyear="
url_year_filter="&birthyearfilter="
url_suffix="&deathyear=&deathyearfilter=&locationId=&includeMaidenName=true"


sleep_range=[5,12]
sleep_time_pause=1*60
cemetery_count=0
total_frame_count=0
total_cemetery_count=len(cemetery_info_df)
get_counter=0
name_count=0
for index, row in cemetery_info_df.iterrows():
    country_name=row['country_name']	
    locationId	=row['locationId']
    cemetery_name=row['cemetery_name']	
    cemetery_id=row['cemetery_id']
    cemetery_count+=1
    cemetery_grave_count=0
    start_year=1890
    last_year=1930
    interval=32
    search_year=start_year+interval
    first_search_year=int(search_year-interval)
    last_search_year=int(search_year+interval)
    while search_year<last_year:
        frame_count=0
        grave_frame_count=0
        grave_count=100
        print(f"\n *** {process_name:<20} • Cemetery: {cemetery_name} • {country_name} • {cemetery_id} •• Next Search Year: {first_search_year}-{search_year}-{last_search_year} ******* ")
        get_counter+=1
        while grave_frame_count<grave_count and interval!=0:
            url=url_cemetery_id+str(cemetery_id)+url_skip+str(grave_frame_count)+url_birth_year+str(search_year)+url_year_filter+str(interval)
            capture_json,grave_count=get_grave_count(url)
            if status_code!=200:print('\tstatus_code:',status_code);break 
            while grave_count>10000 and interval>1:
                interval=int(interval/2)
                search_year-=interval
                first_search_year=int(search_year-interval)
                last_search_year=int(search_year+interval)
                url=url_cemetery_id+str(cemetery_id)+url_skip+str(grave_frame_count)+url_birth_year+str(search_year)+url_year_filter+str(interval)
                capture_json,grave_count=get_grave_count(url)             
            grave_frame_df=extract_grave_data(capture_json)            
            if len(grave_frame_df)==0:
                if (search_year+3*interval)<=last_year:
                    search_year+=2*interval
                else:
                    interval=int((last_year-last_search_year)/2)
                    search_year=last_search_year+interval
                first_search_year=int(search_year-interval)
                last_search_year=int(search_year+interval)
                continue
            grave_frame_df=clean_dataframe(grave_frame_df)
            graves_df = pd.concat([graves_df,grave_frame_df]) 
            grave_frame_count+=len(grave_frame_df)
            cemetery_grave_count+=len(grave_frame_df)

            graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
            print(f" {blank_line}", end='\r')
            print(f"    --- {process_name:<20}  • Cemetery: {cemetery_name} • {country_name} • {cemetery_id} •• Search Year: {first_search_year}-{search_year}-{last_search_year} • grave count: {grave_frame_count}/{grave_count} ", end='\r')
            get_counter+=1
            sleep_time=random_sleep(get_counter)       
            time.sleep(sleep_time) 
        if (search_year+3*interval)<last_year:
            search_year+=2*interval
        else:
            interval=int((last_year-last_search_year)/2)
            search_year=last_search_year+interval
        first_search_year=int(search_year-interval)
        last_search_year=int(search_year+interval)
        if interval==0:break
    print(f"***{process_name:<20} • Cemetery: {cemetery_name} • {country_name} • {cemetery_id} ••  Cemetery Grave Count: {cemetery_grave_count} • Cemetery count: {cemetery_count}/{total_cemetery_count} \n")
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


    


**Starting ***  Extract_ABMC_Graves_FAG  *** , Time: 2025-07-22 09:46:52.986716  **


 *** Extract_ABMC_Graves_FAG • Cemetery: Ardennes American Cemetery • Belgium • 1991676 •• Next Search Year: 1890-1922-1954 ******* 
***Extract_ABMC_Graves_FAG • Cemetery: Ardennes American Cemetery • Belgium • 1991676 ••  Cemetery Grave Count: 4893 • Cemetery count: 1/19 93/4893                                         


 *** Extract_ABMC_Graves_FAG • Cemetery: Henri-Chapelle American Cemetery • Belgium • 1991678 •• Next Search Year: 1890-1922-1954 ******* 
			Sleeping for 60 seconds • Sleeping till: 07/22/2025 10:03:18rican Cemetery • Belgium • 1991678 •• Search Year: 1890-1922-1954 • grave count: 4900/7520                                   
***Extract_ABMC_Graves_FAG • Cemetery: Henri-Chapelle American Cemetery • Belgium • 1991678 ••  Cemetery Grave Count: 7520 • Cemetery count: 2/19 20/7520                                   


 *** Extract_ABMC_Graves_FAG • Cemetery: Brittany American Cemetery • F

In [ ]:
collection_df=pd.json_normalize('collection'])
list(collection_df)

In [ ]:
field_list=['lastName','firstName','middleName','fullName','nickName','birthDate','birthMonth','birthYear','birthCirca','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId', 'intermentHasPhoto','Parents','Spouses','Siblings','Children']